# General Statistics
Here we will explore some basic metrics about text statistics. Not really the state of the art, but it's a good place to start.

In [ ]:
import importlib
import toolbox_gs
importlib.reload(toolbox_gs)


In [ ]:
import os
import spacy
import re
import sys
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from prettytable import PrettyTable
from tqdm import tqdm
import sys
sys.path.append ('..')
from progress.checkpoint import checkpoint
from toolbox_gs import corpus_stats, author_stats, plot_authors, word_frequency, word_frequency_plot, word_frequency_combined, top_words_plot, compare_authors_heatmap, cumulative_coverage_plot,sentence_length_histogram, word_length_histogram, sentence_length_over_text, word_length_over_text, zipf_plot, ttr_curve, punctuation_density

SESSION_ID = 'General Statistics'
checkpoint (SESSION_ID, 'imports')

In [13]:
#############################################################
#                                                           #
#   CONFIGURATION (EDIT THIS CELL TO CHANGE SETTINGS)       #
#                                                           #
#############################################################
input_folder_forms = os.path.expanduser("/home/michauw/Praca/zasoby/korpusy/tekstowe/ENXIX/original")
input_folder_lemmas = os.path.expanduser("/home/michauw/Praca/zasoby/korpusy/tekstowe/ENXIX/lemmatised")
try:
    assert os.path.exists (input_folder_forms)
    assert os.path.exists (input_folder_lemmas)
    checkpoint (SESSION_ID, 'data')
except AssertionError:
    if not os.path.exists (input_folder_forms):
        print (f'The path {input_folder_forms} doesn\'t exist, please correct it')
    if not os.path.exists (input_folder_forms_lemmas):
        print (f'The path {input_folder_forms_lemmas} doesn\'t exist, please correct it')

✅ Checkpoint data for session 'General Statistics' registered successfully!


## Stats per File

In [ ]:
# Chars -> number of characters (excluding whitespace) per text file + aggregate for the whole corpus
# Words -> number of word tokens per text file + aggregate for the whole corpus
# Types -> number of unique word types per text file + aggregate for the whole corpus
# Sentences -> number of sentences per text file + aggregate for the whole corpus
# Avg word -> average word length per text file + aggregate for the whole corpus
# Avg sent -> average sentence length in words per text file + aggregate for the whole corpus
# LD (TTR) -> lexical diversity (type-token ratio) per text file + aggregate for the whole corpus
# MTLD -> Measure of Textual Lexical Diversity per text file + aggregate for the whole corpus
# Hapax % -> percentage of word types that appear exactly once per text file + aggregate for the whole corpus: number of hapax legomena / number of word types * 100
# TTR-1k -> TTR averaged over fixed-size non-overlapping windows of 1000 tokens per text file + aggregate for the whole corpus

corpus_stats(input_folder_forms)
checkpoint (SESSION_ID, 'corpus-stats (word forms)')

### wyjaśnienia kolumn do osobnej komórki MD

Scanning 90 .txt files in /home/michauw/Praca/zasoby/korpusy/tekstowe/ENXIX/original (sample_size=5k) ...

+----------------------------------+------------+------------+----------+-----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
| File                             |      Chars |      Words |    Types | Sentences | Avg word | Avg sent |     MTLD |   TTR-1k |   TTR@5k | Hapax@5k | LD (TTR) |  Hapax % | Warning  |
+----------------------------------+------------+------------+----------+-----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
| C_aubin_charlotta.txt            |    298,552 |     68,194 |    5,420 |     1,436 |     4.23 |    47.49 |     89.3 |   0.4163 |   0.2542 |    60.9% |   0.0795 |    43.9% | OK       |
| C_baker_drawingroom.txt          |     52,220 |     11,067 |    3,040 |       393 |     4.54 |    28.16 |     86.2 |   0.4728 |   0.3334 |    70.2% |  

In [15]:
corpus_stats(input_folder_lemmas)

Scanning 90 .txt files in /home/michauw/Praca/zasoby/korpusy/tekstowe/ENXIX/lemmatised (sample_size=5k) ...

+----------------------------------+------------+------------+----------+-----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
| File                             |      Chars |      Words |    Types | Sentences | Avg word | Avg sent |     MTLD |   TTR-1k |   TTR@5k | Hapax@5k | LD (TTR) |  Hapax % | Warning  |
+----------------------------------+------------+------------+----------+-----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
| C_aubin_charlotta.txt            |    287,362 |     68,194 |    4,748 |     1,436 |     4.07 |    47.49 |     71.0 |   0.3871 |   0.2326 |    60.1% |   0.0696 |    43.1% | OK       |
| C_baker_drawingroom.txt          |     49,726 |     11,067 |    2,629 |       393 |     4.32 |    28.16 |     72.2 |   0.4444 |   0.3024 |    67.1% |

In [16]:
author_stats (input_folder_forms)

Scanning 90 files across 4 authors in /home/michauw/Praca/zasoby/korpusy/tekstowe/ENXIX/original (sample_size=5k) ...

+--------------+----------+------------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
| Author       |    Files |      Words | Avg book | Avg word | Avg sent |     MTLD |   TTR-1k |   TTR@5k | Hapax@5k | LD (TTR) |  Hapax % | Warning  |
+--------------+----------+------------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
| C            |       33 |  3,154,895 |   95,603 |     4.55 |    28.21 |    103.0 |   0.4499 |   0.2542 |    60.9% |   0.0174 |    50.7% | OK       |
| M            |       23 |  3,645,845 |  158,515 |     4.24 |    25.44 |     87.4 |   0.4110 |   0.2268 |    58.3% |   0.0183 |    53.8% | OK       |
| W            |       22 |  3,503,165 |  159,235 |     4.45 |    21.89 |    100.7 |   0.4395 |   0.2382 |    58.9% |   0.0176

In [ ]:
author_stats (input_folder_lemmas)

<div style="
  background: rgba(247, 244, 200, 0.2);
">
<h2>Column Explanations (default data)</h2>

**Files** — Number of .txt files belonging to this author.

**Words** — Total tokens (running word count) across all of the author's files.

**Avg book** — Mean number of words per file. Useful for comparing how long this author's typical work is.
Length-independent style — safe to compare across authors of any size

**Avg word** — Mean word length, in characters. Higher values often suggest a more formal or technical register.

**Avg sent** — Mean sentence length, in words. Higher values often suggest more complex syntax (academic, legal, 19th-century novels).

**MTLD** — Measure of Textual Lexical Diversity. Walks the text and counts how many tokens fit before the running TTR drops below 0.72. Designed to be length-independent. Higher = richer vocabulary.

**TTR-1k** — Mean Type-Token Ratio computed in non-overlapping 1000-word windows. Length-independent variant of plain TTR. Higher = more lexical variety. Fair-sample comparison — identical token budget from every author

**TTR@n** — TTR computed on the first n tokens only. If the author has fewer than n tokens this shows a warning, meaning they cannot be compared on equal footing. The most robust like-for-like comparison in the table.
**Hapax@n** — Percentage of word types appearing exactly once, computed on the same n-token sample. Higher = more vocabulary turnover.
Length-dependent — shown for reference, but biased by corpus size

**LD (TTR)** — Plain Type-Token Ratio: types / words on the full text. Falls steadily as a corpus grows, so an author with 100k words will always score lower than one with 5k, regardless of style.

**Hapax %** — Percentage of word types appearing exactly once across the author's entire corpus. Also drops with corpus size.

**TOTAL row** — Recomputed on the concatenated corpus of every author, not as an average of the rows above. MTLD/TTR/Hapax don't average meaningfully, so the aggregate is built from the underlying tokens.
</div>

<div style="
  background: rgba(247, 244, 240, 0.2);
">
<h2>TTR-1k vs TTR@N — what's the difference?</h2>

Both metrics use the Type-Token Ratio (TTR = unique words / total words), but they answer different questions.

**TTR-1k (windowed mean TTR)**

Walks through the **entire text** of an author and computes TTR in non-overlapping 1000-word windows, then averages across windows.

```
> Example: an author with 7000 words gives 7 windows → 7 TTR values → mean.
```

The number 1k is the window size: how many tokens to look at at a time before averaging. It's a smoothing parameter, not a sample size. Every word the author wrote contributes to the result.

Question it answers: *"How varied is this author's vocabulary on average, throughout their text?"*

**TTR@N (fair-sample TTR)**

Takes only the first **N** tokens from each author and computes a single TTR on that slice. The rest of the text is ignored.

```
Example: an author with 7000 words and sample_size=5000 → TTR computed on words 1–5000, words 5001–7000 discarded.
```

The number N is the sample size: every author is compared on the same token budget. If an author has fewer than N tokens, the metric is computed on whatever they have (and the Warning column flags them).

Question it answers: *"On the same fixed amount of text from each author, how varied is the vocabulary?"*

**Why TTR-1k is not affected by sample_size?**

sample_size controls how big a slice we cut for the fair-comparison metrics (TTR@N, Hapax@N). It is unrelated to the smoothing window of TTR-1k, which is fixed at 1000 tokens by design. The two parameters belong to different families of measurements:

Smoothing window → applied to the whole text, used to make TTR less noisy
Sample size → applied to a fixed prefix, used to make authors directly comparable
When to use which

**TTR-1k** — when you want one number summarising lexical variety across an author's whole corpus

**TTR@N** — when you want a strict, like-for-like comparison between authors of different corpus sizes

**MTLD** — when you want a single length-independent number that doesn't require choosing a window or a sample (different algorithm; recommended as the default lexical-diversity metric)

## Summary

| Metric | What it measures | Uses how much text | Affected by `sample_size`? |
|--------|------------------|-------------------|----------------------------|
| **TTR (LD)** | Variety of vocabulary on the full text | All of it | ⚪ No |
| **TTR-1k** | Average variety in 1000-word windows | All of it (averaged) | ⚪ No |
| **TTR@N** | Variety on the first N tokens (fair sample) | First N tokens only | 🔵 Yes |
| **Hapax %** | % of word types appearing exactly once on full text | All of it | ⚪ No |
| **Hapax@N** | Hapax % on the first N tokens (fair sample) | First N tokens only | 🔵 Yes |
| **MTLD** | Length-independent lexical diversity (segments to TTR=0.72) | All of it | ⚪ No |
| **Avg word** | Mean word length in characters | All of it | ⚪ No |
| **Avg sent** | Mean sentence length in words | All of it | ⚪ No |
| **Avg book** | Mean words per file (per book) | All of it | ⚪ No |

**Takeaway:** Only **`TTR@N`** and **`Hapax@N`** change when you change **`sample_size`**. Everything else uses the full text of each author.
</div>

## Beyond averages: distributions and trajectories
Beware! Descriptive statistics like *"Average sentence length = 18 words" are useful, but they hide enormous amounts of structure.

Two texts with identical mean sentence length can read completely differently — one alternates rapid dialogue and long descriptions, the other plods along uniformly. 

### ***Each plot adds one more observation about the same data!!!***

To see that we look at:

### Distributions (histograms)
- **`sentence_length_histogram(file)`** - how varied are this author's sentences? A tall narrow history = uniform style; a wide or bimodal one = mixed register
- **`word_length_histogram(file)`** - characteristic shape for English peaks at 3-4 characters. Watch for long tails: technical vocabulary, archaic forms, or formal register. 

### ***Sentence Length Histogram:***

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Austen_Emma.txt")
fig = sentence_length_histogram(input_file)

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/EBronte_Wuthering.txt")
fig = sentence_length_histogram(input_file)

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Richardson_Clarissa.txt")
fig = sentence_length_histogram(input_file)

### **Word length histogram**

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Richardson_Clarissa.txt")
fig = word_length_histogram(input_file)

## Trajectories (over the text)
- **`sentence_length_over_text(file, window=20)`** - rolling mean of sentence length accross the book. Peaks usually mark descriptive or contemplative passages; troughs mark dialogue or action.
- **`word_length_over_text(file, chunk_size=1000)`** - register shifts visible as up/down jumps. Useful for finding the boundary between, say, a scientific preface and a narrative.

## Reading the trajectory plots

Both `sentence_length_over_text` and `word_length_over_text` show how a measure changes **across the text**, from start to end. They help you see where in the book the style shifts — without reading it.

### `sentence_length_over_text(file, window=20)`

Each point on the line represents **one sentence**, with the y-value being the **mean length of `window` sentences centred on that point**.

- **X-axis**: sentence index (1, 2, 3, ...)
- **Y-axis**: rolling average of sentence length in words
- **One point per sentence**

With `window=20`, sentence number 50 shows the average of sentences 40–60. Smaller window = less smoothing, more local noise. Larger window = smoother curve, only big structural moves are visible. Set `window=1` if you want raw lengths with no smoothing.

**What to look for:** dips usually mark dialogue or rapid action ("Yes." "No." "Are you sure?"), peaks mark descriptive or contemplative passages with longer constructions. A flat line means uniform pacing.

### `word_length_over_text(file, chunk_size=1000)`

Each point represents one **non-overlapping chunk** of `chunk_size` tokens — not one sentence.

- **X-axis**: token position in the text (the chunk's centre)
- **Y-axis**: mean word length (characters) in that chunk
- **Number of points**: `total_tokens / chunk_size`

For a 80 000-word novel with `chunk_size=1000`, you get about 80 points. Each is plotted at the **middle** of its chunk (the first at position 500, the next at 1500, and so on).

**What to look for:** rises indicate longer-word passages — technical or formal register, descriptive prose, scientific terminology. Drops indicate sections heavy in short words — dialogue, action, simple narration. The red dashed line is the overall mean for comparison.

### Why the two work differently

Sentences are natural discrete units, so a rolling window of *N sentences* is intuitive. Words are too granular for that — 80 000 rolling points per book would be unreadable — so we group words into bigger chunks and plot one point per chunk instead. The visual idea is the same: smooth a noisy signal so the underlying structure becomes visible.

### Choosing the parameter

| | Small | Medium (default) | Large |
|---|---|---|---|
| `sentence_length_over_text` window | `5` — see chapter-by-chapter | `20` | `50` — see book-wide trends only |
| `word_length_over_text` chunk_size | `200` — micro register shifts | `1000` | `5000` — coarse trends |

### A quick experiment

Try both on the same novel:

```python
sentence_length_over_text("given_file.txt", window=20)
word_length_over_text("data/given_file", chunk_size=1000)
```

You'll often see the two curves move together (a dialogue section has both short sentences *and* short words) but not always — descriptive prose can have long sentences full of short words, and a technical passage can have short sentences full of long words. The combination tells you more than either alone.


### **Sentence length over text**

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/EBronte_Wuthering.txt")
fig = sentence_length_over_text(input_file, window=20)

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Richardson_Clarissa.txt")
fig = sentence_length_over_text(input_file, window=20)

### **Word length over text**

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/EBronte_Wuthering.txt")
fig = word_length_over_text(input_file)

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Richardson_Clarissa.txt")
fig = word_length_over_text(input_file)

## Stats per Author

In [ ]:
input_folder = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus")
author_stats(input_folder, sample_size=200)

## Authors' plot

In [ ]:
fig = plot_authors(
    input_folder,
    x_metric="hapax_n",
    y_metric="mtld",
    size_metric="words",
    sample_size=10000,
    title="Authors in Hapax % vs. MTLD space (point size \u221d number of books)",
)

## Word frequencies

In [ ]:
word_frequency(input_folder, ["love", "death", "god"])

In [ ]:
fig = word_frequency_combined(input_folder, ["love", "death", "god"], chunk_size=1000)

In [ ]:
fig = word_frequency_combined(input_folder, ["love", "death", "god"], chunk_size=10000)

In [ ]:
fig = word_frequency_plot(input_folder, ["love", "death", "god"], chunk_size=5000)

## How to lemmatize texts?
If you need to lemmatize your corpus, this can be helpful:

In [ ]:
def lemmatize_text(nlp, text, chunk_size=200000):
    # Process large texts in chunks to avoid spaCy limits
    parts = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        doc = nlp(chunk)

        parts.append("".join(
            (token.text if token.lemma_ == "-PRON-" else token.lemma_) + token.whitespace_
            for token in doc
        ))

    return "".join(parts)


def process_folder(input_dir, output_dir, extension=".txt"):
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Load spaCy model (faster without parser/NER)
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

    # Collect all files first for progress bar
    all_files = []

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if extension and not file.endswith(extension):
                continue

            all_files.append((root, file))

    # Progress bar over all files
    for root, file in tqdm(all_files, desc="Lemmatizing corpus"):
        relative_path = os.path.relpath(root, input_dir)
        target_root = os.path.join(output_dir, relative_path)
        os.makedirs(target_root, exist_ok=True)

        input_file = os.path.join(root, file)
        output_file = os.path.join(target_root, f"{os.path.splitext(file)[0]}_lemmatized{os.path.splitext(file)[1]}")

        # Read file
        with open(input_file, "r", encoding="utf-8") as f:
            text = f.read()

        # Lemmatize
        lemmatized = lemmatize_text(nlp, text)

        # Write output
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(lemmatized)

In [ ]:
# And now let's lemmatize it!
### Beware: this will take a while, especially if you have a lot of files. You can interrupt it with Ctrl+C if needed.
### For the whole corpus it may take around 15 minutes, but it depends on your machine.

input_folder = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/")
output_folder = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/lemmatized_corpus/")

process_folder(input_folder, output_folder)

## Stop-Words (Function Words)

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Austen_Emma.txt")
fig = top_words_plot(input_file, n=30)

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Austen_Emma.txt")
fig = cumulative_coverage_plot(input_file, max_n=500)

In [ ]:
input_folder = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/")
fig = compare_authors_heatmap(input_folder, n=30) 

In [ ]:
input_folder = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/")
fig = compare_authors_heatmap(input_folder, n=30, exclude_stop_words=True) 

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Austen_Emma.txt")
fig = zipf_plot(input_file)

In [ ]:
input_file = os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/corpus/Austen_Emma.txt")
fig = ttr_curve(input_file)

# **Conclusions for this day:**
### **TEXT IS NOT NOISE**.
### It is deeply **STRUCTURED**, and that structure is **MEASURABLE**.